### Завдання 1: Завантаження даних
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу до його імені потрібно додати дату та час завантаження. Реалізувати механізм запобігання повторного довантаження та колізії даних.

In [ ]:
# пе
import urllib.request
import urllib.error
import datetime
import os

data_dir = "vhi_data"
os.makedirs(data_dir, exist_ok=True)

region_names = {
    1: 'Черкаська', 2: 'Чернігівська', 3: 'Чернівецька', 4: 'Крим', 5: 'Дніпропетровська',
    6: 'Донецька', 7: 'Івано-Франківська', 8: 'Харківська', 9: 'Херсонська', 10: 'Хмельницька',
    11: 'Київська', 12: 'м. Київ', 13: 'Кіровоградська', 14: 'Луганська', 15: 'Львівська',
    16: 'Миколаївська', 17: 'Одеська', 18: 'Полтавська', 19: 'Рівненська', 20: 'Севастополь',
    21: 'Сумська', 22: 'Тернопільська', 23: 'Закарпатська', 24: 'Вінницька', 25: 'Волинська',
    26: 'Запорізька', 27: 'Житомирська'
}

def download_vhi_data():
    for pid in range(1, 28):
        name = region_names.get(pid, str(pid))
        
        existing = [f for f in os.listdir(data_dir) if f"vhi_id_{pid:02d}_" in f]
        if existing:
            print(f"{pid} ({name}): вже є")
            continue
        
        url = (
            f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"
            f"?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean"
        )
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filepath = os.path.join(data_dir, f"vhi_id_{pid:02d}_{timestamp}.csv")
        
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"завантажено {pid}: {name}")
        except urllib.error.URLError as e:
            print(f"помилка {name}: {e}")

download_vhi_data()

1 (Черкаська): вже є
2 (Чернігівська): вже є
3 (Чернівецька): вже є
4 (Крим): вже є
5 (Дніпропетровська): вже є
6 (Донецька): вже є
7 (Івано-Франківська): вже є
8 (Харківська): вже є
9 (Херсонська): вже є
10 (Хмельницька): вже є
11 (Київська): вже є
12 (м. Київ): вже є
13 (Кіровоградська): вже є
14 (Луганська): вже є
15 (Львівська): вже є
16 (Миколаївська): вже є
17 (Одеська): вже є
18 (Полтавська): вже є
19 (Рівненська): вже є
20 (Севастополь): вже є
21 (Сумська): вже є
22 (Тернопільська): вже є
23 (Закарпатська): вже є
24 (Вінницька): вже є
25 (Волинська): вже є
26 (Запорізька): вже є
27 (Житомирська): вже є


### 2. Очистка даних та зміна індексів
**Завдання:** Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, видалити зайвий текст, заповнити пропуски. Додати стовпчик з індексом області. Реалізувати процедуру зміни індексів, щоб області індексувалися за українською абеткою.

In [2]:
import pandas as pd
import os

index_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

def load_data(data_dir):
    frames = []
    
    for fname in os.listdir(data_dir):
        if not fname.endswith(".csv"):
            continue
            
        old_id = int(fname.split("_")[2])
        new_id = index_mapping.get(old_id, old_id)
        
        df = pd.read_csv(
            os.path.join(data_dir, fname),
            index_col=False,
            header=1,
            names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        )
        
        df = df.drop(columns=['empty'], errors='ignore')
        df = df.dropna(subset=['Year'])
        df = df[~df['Year'].astype(str).str.contains('<', na=False)]
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        df['VHI'] = df['VHI'].astype(float)
        df['Province_ID'] = new_id
        
        frames.append(df)
    
    result = pd.concat(frames, ignore_index=True)
    result = result[result['VHI'] >= 0]
    return result

df = load_data(data_dir)
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID
0,1982,2,0.034,251.07,40.82,71.85,56.33,11
1,1982,3,0.032,252.56,38.21,64.53,51.37,11
2,1982,4,0.030,254.76,32.74,55.45,44.10,11
3,1982,5,0.027,256.18,26.61,53.62,40.12,11
4,1982,6,0.025,256.78,20.92,56.90,38.91,11


### 3.1 Формування вибірки: Ряд VHI за рік
**Завдання:** Реалізувати процедуру для формування вибірки (Ряд VHI для області за вказаний рік). 

In [3]:
def get_vhi_series(df, province_id, year):
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    return result[['Week', 'VHI']].reset_index(drop=True)

get_vhi_series(df, 1, 2020).head()

,Week,VHI
0,1,40.92
1,2,43.19
2,3,44.74
3,4,45.29
4,5,44.80


### 3.2. Вибірка: Ряд VHI за діапазон років для вказаних областей
**Завдання:** Реалізувати процедуру для формування вибірки (ряд VHI за вказаний діапазон років для вказаних областей).

In [4]:
def get_vhi_range(df, provinces, year_from, year_to):
    mask = (
        df['Province_ID'].isin(provinces) &
        (df['Year'] >= year_from) &
        (df['Year'] <= year_to)
    )
    return df[mask][['Year', 'Week', 'Province_ID', 'VHI']].reset_index(drop=True)

get_vhi_range(df, [22, 24], 2015, 2017).head(10)

,Year,Week,Province_ID,VHI
0,2015,1,24,50.69
1,2015,2,24,50.77
2,2015,3,24,48.43
3,2015,4,24,47.89
4,2015,5,24,47.91
5,2015,6,24,46.76
6,2015,7,24,44.75
7,2015,8,24,41.95
8,2015,9,24,39.44
9,2015,10,24,37.94


### 3.3. Пошук екстремумів, середнього та медіани
**Завдання:** Реалізувати процедуру для пошуку екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [5]:
def get_vhi_stats(df, provinces, year_from, year_to):
    mask = (
        df['Province_ID'].isin(provinces) & 
        (df['Year'] >= year_from) & 
        (df['Year'] <= year_to)
    )
    vhi_data = df[mask]['VHI']
    
    return pd.Series({
        'Мінімум (min)': vhi_data.min(),
        'Максимум (max)': vhi_data.max(),
        'Середнє (mean)': round(vhi_data.mean(), 2),
        'Медіана (median)': vhi_data.median()
    })
get_vhi_stats(df, [22, 24], 2015, 2017)

Мінімум (min)       34.290
Максимум (max)      68.990
Середнє (mean)      47.460
Медіана (median)    46.375
dtype: float64